# Modul 18: LeNet und Transfer Learning mit PyTorch

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** LeNet mit PyTorch, Transfer mit PyTorch  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene PyTorch-Anwendung für Bilddaten  
    **Orientierungszeit:** etwa 160 bis 220 Minuten

    ## Überblick

    Sie bauen eine Bilddatenpipeline, implementieren ein LeNet-ähnliches nn.Module und analysieren Training sowie Fehlerbilder. Danach untersuchen Sie Augmentation, Dropout, BatchNorm, Scheduler und ein kleines Transfer-Learning-Experiment mit MobileNetV3 Small.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_18A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_18B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Bildtransformationen und DataLoader für Trainings-, Validierungs- und Testbilder definieren.
- Ausgabeformen von Conv2d, Pooling und Flatten im channels-first-Format bestimmen.
- Eine LeNet-Klasse mit nn.Module implementieren und auf CPU oder GPU trainieren.
- Fehlerbilder und Framework-spezifische Datenformen nachvollziehbar analysieren.
- Augmentation, Dropout, BatchNorm, Lernratenscheduler und Early Stopping kombinieren.
- MobileNetV3 Small laden, Feature-Schichten einfrieren und einen Klassifikationskopf anpassen.
- Genauigkeit, trainierbare Parameter, Inferenzzeit und Artefaktgröße gemeinsam bewerten.

    ## Bewertete Fähigkeiten

    - torchvision.transforms, Dataset und DataLoader
- Conv2d, MaxPool2d, Flatten und LeNet als nn.Module
- CNN-Training und Fehleranalyse
- Augmentation, BatchNorm, Dropout, Scheduler und Early Stopping
- MobileNetV3 Small, Freezing, Transfer-Kopf und Ressourcenvergleich

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames PyTorch-Setup
import io
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)
warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torchvision import models, transforms

digits_18 = load_digits()
images_18 = (digits_18.images.astype("float32") / 16.0)
labels_18 = digits_18.target.astype("int64")

X_train_valid_18, X_test_18, y_train_valid_18, y_test_18 = train_test_split(
    images_18,
    labels_18,
    test_size=0.20,
    stratify=labels_18,
    random_state=RANDOM_SEED,
)
X_train_18, X_valid_18, y_train_18, y_valid_18 = train_test_split(
    X_train_valid_18,
    y_train_valid_18,
    test_size=0.25,
    stratify=y_train_valid_18,
    random_state=RANDOM_SEED,
)

transfer_mask_18 = np.isin(labels_18, [0, 1, 2])
transfer_images_18 = images_18[transfer_mask_18]
transfer_labels_18 = labels_18[transfer_mask_18]
TX_train_valid_18, TX_test_18, Ty_train_valid_18, Ty_test_18 = train_test_split(
    transfer_images_18,
    transfer_labels_18,
    test_size=0.20,
    stratify=transfer_labels_18,
    random_state=RANDOM_SEED,
)
TX_train_18, TX_valid_18, Ty_train_18, Ty_valid_18 = train_test_split(
    TX_train_valid_18,
    Ty_train_valid_18,
    test_size=0.25,
    stratify=Ty_train_valid_18,
    random_state=RANDOM_SEED,
)

print("LeNet Train/Valid/Test:", X_train_18.shape, X_valid_18.shape, X_test_18.shape)
print("Transfer Train/Valid/Test:", TX_train_18.shape, TX_valid_18.shape, TX_test_18.shape)

print("PyTorch-Version:", torch.__version__)
print("Gerät:", DEVICE)


## Aufgabe 1: Bild-Dataset, Transformationen und Conv-Formen

    Erstellen Sie eine kleine PyTorch-Bilddatenpipeline.

1. Definieren Sie eine `Dataset`-Klasse, die NumPy-Bilder und Labels speichert.
2. Wandeln Sie jedes `8 x 8`-Bild in einen `float32`-Tensor der Form `(1, 8, 8)` um.
3. Erstellen Sie getrennte DataLoader. Mischen Sie nur das Training.
4. Prüfen Sie einen Batch auf die Form `(Batch, Kanäle, Höhe, Breite)`.
5. Wenden Sie testweise `Conv2d(1, 6, kernel_size=3, padding=1)` und `MaxPool2d(2)` an und bestätigen Sie die resultierenden Formen.
6. Erklären Sie den Formunterschied zum standardmäßigen Keras-Format.

> **Hinweis:** PyTorch erwartet die Kanalachse vor den räumlichen Achsen.

In [ ]:
batch_size_18 = 32

# Speichern Sie die Loader als train_loader_18, valid_loader_18 und
# test_loader_18 für die folgenden Aufgaben.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Bild-Dataset, Transformationen und Conv-Formen
#
# Ziel dieser Codezelle:
# Erstellen Sie eine kleine PyTorch-Bilddatenpipeline. 1. Definieren Sie eine
# Dataset-Klasse, die NumPy-Bilder und Labels speichert. 2. Wandeln Sie jedes 8 x
# 8-Bild in einen float32-Tensor der Form (1, 8, 8) um. 3. Erst...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

batch_size_18 = 32

class DigitsDataset18(Dataset):
    def __init__(self, images, labels, transform=None):
        if len(images) != len(labels):
            raise ValueError("Bilder und Labels müssen gleich lang sein.")
        self.images = np.asarray(images, dtype=np.float32)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # unsqueeze ergänzt die Kanalachse vor Höhe und Breite.
        image = torch.tensor(self.images[index], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.labels[index], dtype=torch.long)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset_18 = DigitsDataset18(X_train_18, y_train_18)
valid_dataset_18 = DigitsDataset18(X_valid_18, y_valid_18)
test_dataset_18 = DigitsDataset18(X_test_18, y_test_18)

loader_generator_18 = torch.Generator().manual_seed(RANDOM_SEED)
train_loader_18 = DataLoader(
    train_dataset_18,
    batch_size=batch_size_18,
    shuffle=True,
    generator=loader_generator_18,
    num_workers=0,
)
valid_loader_18 = DataLoader(
    valid_dataset_18,
    batch_size=batch_size_18,
    shuffle=False,
    num_workers=0,
)
test_loader_18 = DataLoader(
    test_dataset_18,
    batch_size=batch_size_18,
    shuffle=False,
    num_workers=0,
)

first_images_18, first_labels_18 = next(iter(train_loader_18))
assert first_images_18.shape == (batch_size_18, 1, 8, 8)
assert first_labels_18.shape == (batch_size_18,)
assert first_images_18.dtype == torch.float32
assert first_labels_18.dtype == torch.int64

# Eine same-artige Faltung mit padding=1 hält 8 x 8. Pooling halbiert
# anschließend beide räumlichen Achsen.
test_conv_18 = nn.Conv2d(1, 6, kernel_size=3, padding=1)
test_pool_18 = nn.MaxPool2d(kernel_size=2, stride=2)
conv_output_18 = test_conv_18(first_images_18)
pooled_output_18 = test_pool_18(conv_output_18)
assert conv_output_18.shape == (batch_size_18, 6, 8, 8)
assert pooled_output_18.shape == (batch_size_18, 6, 4, 4)

print("Eingabebatch:", tuple(first_images_18.shape))
print("Nach Conv2d:", tuple(conv_output_18.shape))
print("Nach MaxPool2d:", tuple(pooled_output_18.shape))

### Reflexion zu Aufgabe 1

PyTorch verwendet standardmäßig `(Batch, Kanäle, Höhe, Breite)`, während Keras häufig `(Batch, Höhe, Breite, Kanäle)` nutzt. Eine Verwechslung führt zu falschen Kanalzahlen oder Formfehlern. Transformationen gehören in die Dataset-Pipeline, sodass sie für jedes geladene Beispiel kontrolliert angewendet werden. Shuffling ist eine Trainingsoperation und für deterministische Fehleranalyse auf Testdaten nicht nötig.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: LeNet als nn.Module implementieren

    Implementieren Sie ein LeNet-ähnliches CNN für zehn Klassen.

1. Erstellen Sie zwei Faltungsblöcke mit ReLU und Max-Pooling.
2. Bestimmen Sie die Flatten-Größe durch eine kurze Formrechnung oder einen sicheren Hilfsdurchlauf.
3. Ergänzen Sie eine Dense-Schicht und eine lineare Ausgabeschicht mit zehn Logits.
4. Verschieben Sie das Modell auf `DEVICE`.
5. Richten Sie `CrossEntropyLoss` und Adam ein.
6. Prüfen Sie Ausgabeform und Parameterzahl.

> **Hinweis:** Verfolgen Sie die Form `1x8x8 -> 8x4x4 -> 16x2x2`.

In [ ]:
# Speichern Sie die Objekte als lenet_18, criterion_18 und optimizer_18.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: LeNet als nn.Module implementieren
#
# Ziel dieser Codezelle:
# Implementieren Sie ein LeNet-ähnliches CNN für zehn Klassen. 1. Erstellen Sie zwei
# Faltungsblöcke mit ReLU und Max-Pooling. 2. Bestimmen Sie die Flatten-Größe durch
# eine kurze Formrechnung oder einen sicheren Hilfsdur...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

class LeNet18(nn.Module):
    def __init__(self, number_of_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # 8 x 8 wird nach zwei Pooling-Schritten zu 2 x 2.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, number_of_classes),
        )

    def forward(self, images):
        feature_maps = self.features(images)
        return self.classifier(feature_maps)

torch.manual_seed(RANDOM_SEED)
lenet_18 = LeNet18(number_of_classes=10).to(DEVICE)
criterion_18 = nn.CrossEntropyLoss()
optimizer_18 = torch.optim.Adam(lenet_18.parameters(), lr=0.001)

example_images_18 = first_images_18.to(DEVICE)
example_logits_18 = lenet_18(example_images_18)
parameter_count_18 = sum(
    parameter.numel()
    for parameter in lenet_18.parameters()
    if parameter.requires_grad
)
assert example_logits_18.shape == (batch_size_18, 10)

print(lenet_18)
print("Logitform:", tuple(example_logits_18.shape))
print("Trainierbare Parameter:", parameter_count_18)

### Reflexion zu Aufgabe 2

`CrossEntropyLoss` erwartet rohe Logits der Form `(Batch, Klassen)` und ganzzahlige Klassenindizes. Eine Softmax-Schicht gehört deshalb nicht in die `forward`-Methode für das Training. Für Wahrscheinlichkeiten bei der Interpretation kann später `torch.softmax(logits, dim=1)` verwendet werden. Die Flatten-Größe muss aus Kanalzahl und räumlichen Dimensionen nach allen Faltungsblöcken folgen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: CNN trainieren und Fehlerbilder untersuchen

    Trainieren und bewerten Sie `lenet_18`.

1. Schreiben Sie Trainings- und Evaluationsfunktionen für Mehrklassenklassifikation.
2. Trainieren Sie mit Early Stopping anhand des Validierungsverlusts.
3. Stellen Sie die besten Gewichte wieder her.
4. Visualisieren Sie Loss und Accuracy.
5. Berechnen Sie Testgenauigkeit und Konfusionsmatrix.
6. Zeigen Sie bis zu sechs falsch klassifizierte Bilder mit wahrem Label, Vorhersage und Konfidenz.

> **Hinweis:** Speichern Sie die besten Parameter als unabhängige CPU-Kopien.

In [ ]:
max_epochs_18 = 5 if FAST_MODE else 35
patience_18 = 6

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: CNN trainieren und Fehlerbilder untersuchen
#
# Ziel dieser Codezelle:
# Trainieren und bewerten Sie lenet18. 1. Schreiben Sie Trainings- und
# Evaluationsfunktionen für Mehrklassenklassifikation. 2. Trainieren Sie mit Early
# Stopping anhand des Validierungsverlusts. 3. Stellen Sie die besten...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

max_epochs_18 = 5 if FAST_MODE else 35
patience_18 = 6

def train_multiclass_epoch_18(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.shape[0]
        predictions = logits.argmax(dim=1)
        total_loss += float(loss.item()) * batch_size
        total_correct += int((predictions == labels).sum().item())
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples

def evaluate_multiclass_18(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    probability_parts = []
    label_parts = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += float(loss.item()) * images.shape[0]
            probability_parts.append(torch.softmax(logits, dim=1).cpu())
            label_parts.append(labels.cpu())

    probabilities = torch.cat(probability_parts).numpy()
    labels = torch.cat(label_parts).numpy()
    predictions = probabilities.argmax(axis=1)
    return (
        total_loss / len(loader.dataset),
        accuracy_score(labels, predictions),
        probabilities,
        labels,
    )

history_18 = {
    "train_loss": [],
    "train_accuracy": [],
    "valid_loss": [],
    "valid_accuracy": [],
}
best_valid_loss_18 = np.inf
best_state_18 = None
no_improvement_18 = 0

for epoch in range(max_epochs_18):
    train_loss_18, train_accuracy_18 = train_multiclass_epoch_18(
        lenet_18,
        train_loader_18,
        criterion_18,
        optimizer_18,
    )
    valid_loss_18, valid_accuracy_18, _, _ = evaluate_multiclass_18(
        lenet_18,
        valid_loader_18,
        criterion_18,
    )
    history_18["train_loss"].append(train_loss_18)
    history_18["train_accuracy"].append(train_accuracy_18)
    history_18["valid_loss"].append(valid_loss_18)
    history_18["valid_accuracy"].append(valid_accuracy_18)

    if valid_loss_18 < best_valid_loss_18 - 1e-4:
        best_valid_loss_18 = valid_loss_18
        best_state_18 = {
            name: value.detach().cpu().clone()
            for name, value in lenet_18.state_dict().items()
        }
        no_improvement_18 = 0
    else:
        no_improvement_18 += 1
    if no_improvement_18 >= patience_18:
        break

lenet_18.load_state_dict(best_state_18)
lenet_18.to(DEVICE)
test_loss_18, test_accuracy_18, test_probabilities_18, test_labels_18 = evaluate_multiclass_18(
    lenet_18,
    test_loader_18,
    criterion_18,
)
test_predictions_18 = test_probabilities_18.argmax(axis=1)
confusion_18 = confusion_matrix(test_labels_18, test_predictions_18)

history_frame_18 = pd.DataFrame(history_18)
for metric_name, label in [("loss", "Loss"), ("accuracy", "Accuracy")]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_frame_18[f"train_{metric_name}"], label="Training")
    ax.plot(history_frame_18[f"valid_{metric_name}"], label="Validierung")
    ax.set_title(f"LeNet mit PyTorch: {label}")
    ax.set_xlabel("Epoche")
    ax.set_ylabel(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("Testverlust:", round(float(test_loss_18), 4))
print("Testgenauigkeit:", round(float(test_accuracy_18), 4))
print("Konfusionsmatrix:\n", confusion_18)

error_indices_18 = np.flatnonzero(test_predictions_18 != test_labels_18)[:6]
if len(error_indices_18) > 0:
    fig, axes = plt.subplots(1, len(error_indices_18), figsize=(2.3 * len(error_indices_18), 2.8))
    axes = np.atleast_1d(axes)
    for axis, index in zip(axes, error_indices_18):
        confidence = float(test_probabilities_18[index].max())
        axis.imshow(X_test_18[index], cmap="gray")
        axis.set_title(
            f"wahr {test_labels_18[index]}\nvorh. {test_predictions_18[index]}\np={confidence:.2f}"
        )
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Keine falsch klassifizierten Testbilder gefunden.")

### Reflexion zu Aufgabe 3

Die Konfusionsmatrix zeigt systematische Verwechslungen, während Fehlerbilder konkrete Formen sichtbar machen. Hohe Konfidenz bei falschen Vorhersagen weist darauf hin, dass Softmax-Wahrscheinlichkeiten nicht automatisch kalibrierte Sicherheit bedeuten. Vergleichbare Keras- und PyTorch-Modelle sollten dieselben Splits, Metriken und möglichst ähnliche Architekturen verwenden, bevor Framework-Unterschiede interpretiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Augmentation, BatchNorm, Dropout und Scheduler kombinieren

    Erstellen Sie eine regulierte CNN-Variante.

1. Definieren Sie eine kleine `RandomAffine`-Transformation nur für das Training.
2. Erstellen Sie einen neuen Trainings-DataLoader mit dieser Transformation.
3. Ergänzen Sie BatchNorm nach den Faltungen und Dropout vor der Ausgabeschicht.
4. Verwenden Sie Adam und `ReduceLROnPlateau` auf dem Validierungsverlust.
5. Implementieren Sie weiterhin Early Stopping und speichern Sie die Lernrate pro Epoche.
6. Vergleichen Sie beste Validierungsgenauigkeit und Testgenauigkeit mit dem ersten LeNet-Modell.

> **Hinweis:** Rufen Sie den Scheduler nach der Validierungsphase mit dem überwachten Wert auf.

In [ ]:
regularized_epochs_18 = 5 if FAST_MODE else 35

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Augmentation, BatchNorm, Dropout und Scheduler kombinieren
#
# Ziel dieser Codezelle:
# Erstellen Sie eine regulierte CNN-Variante. 1. Definieren Sie eine kleine
# RandomAffine-Transformation nur für das Training. 2. Erstellen Sie einen neuen
# Trainings-DataLoader mit dieser Transformation. 3. Ergänzen Sie...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

regularized_epochs_18 = 5 if FAST_MODE else 35

# Kleine affine Änderungen simulieren plausible Variationen. Die
# Validierungs- und Testdaten bleiben unverändert.
augmentation_transform_18 = transforms.RandomAffine(
    degrees=8,
    translate=(0.08, 0.08),
    scale=(0.95, 1.05),
)
augmented_train_dataset_18 = DigitsDataset18(
    X_train_18,
    y_train_18,
    transform=augmentation_transform_18,
)
augmented_generator_18 = torch.Generator().manual_seed(RANDOM_SEED + 1)
augmented_train_loader_18 = DataLoader(
    augmented_train_dataset_18,
    batch_size=batch_size_18,
    shuffle=True,
    generator=augmented_generator_18,
    num_workers=0,
)

class RegularizedLeNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, 10),
        )

    def forward(self, images):
        return self.classifier(self.features(images))

torch.manual_seed(RANDOM_SEED + 1)
regularized_lenet_18 = RegularizedLeNet18().to(DEVICE)
regularized_optimizer_18 = torch.optim.Adam(regularized_lenet_18.parameters(), lr=0.001)
scheduler_18 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    regularized_optimizer_18,
    mode="min",
    factor=0.5,
    patience=2,
)

regularized_history_18 = {
    "train_loss": [],
    "valid_loss": [],
    "valid_accuracy": [],
    "learning_rate": [],
}
regularized_best_loss_18 = np.inf
regularized_best_state_18 = None
regularized_no_improvement_18 = 0

for epoch in range(regularized_epochs_18):
    train_loss_regularized_18, _ = train_multiclass_epoch_18(
        regularized_lenet_18,
        augmented_train_loader_18,
        criterion_18,
        regularized_optimizer_18,
    )
    valid_loss_regularized_18, valid_accuracy_regularized_18, _, _ = evaluate_multiclass_18(
        regularized_lenet_18,
        valid_loader_18,
        criterion_18,
    )
    scheduler_18.step(valid_loss_regularized_18)

    regularized_history_18["train_loss"].append(train_loss_regularized_18)
    regularized_history_18["valid_loss"].append(valid_loss_regularized_18)
    regularized_history_18["valid_accuracy"].append(valid_accuracy_regularized_18)
    regularized_history_18["learning_rate"].append(
        regularized_optimizer_18.param_groups[0]["lr"]
    )

    if valid_loss_regularized_18 < regularized_best_loss_18 - 1e-4:
        regularized_best_loss_18 = valid_loss_regularized_18
        regularized_best_state_18 = {
            name: value.detach().cpu().clone()
            for name, value in regularized_lenet_18.state_dict().items()
        }
        regularized_no_improvement_18 = 0
    else:
        regularized_no_improvement_18 += 1
    if regularized_no_improvement_18 >= 6:
        break

regularized_lenet_18.load_state_dict(regularized_best_state_18)
regularized_lenet_18.to(DEVICE)
_, regularized_test_accuracy_18, _, _ = evaluate_multiclass_18(
    regularized_lenet_18,
    test_loader_18,
    criterion_18,
)

regularization_comparison_18 = pd.DataFrame(
    {
        "model": ["LeNet", "reguliertes LeNet"],
        "best_validation_accuracy": [
            max(history_18["valid_accuracy"]),
            max(regularized_history_18["valid_accuracy"]),
        ],
        "test_accuracy": [test_accuracy_18, regularized_test_accuracy_18],
        "parameters": [
            sum(p.numel() for p in lenet_18.parameters()),
            sum(p.numel() for p in regularized_lenet_18.parameters()),
        ],
    }
)
print(regularization_comparison_18.round(4).to_string(index=False))
print("Lernratenverlauf:", regularized_history_18["learning_rate"])

### Reflexion zu Aufgabe 4

Augmentation verändert die tatsächlich gesehenen Trainingsbilder, BatchNorm stabilisiert Aktivierungsstatistiken und Dropout wirkt nur im Trainingsmodus. Der Scheduler reduziert die Lernrate, wenn sich der Validierungsverlust nicht verbessert. Diese Techniken lösen unterschiedliche Probleme und sollten nicht automatisch als Gesamtpaket übernommen werden. Ein fairer Vergleich hält Datenpartition, Metrik und Trainingsbudget konstant.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integration: MobileNetV3 Small einfrieren und bewerten

    Führen Sie ein kleines Transfer-Learning-Projekt für die Klassen 0, 1 und 2 durch.

1. Erstellen Sie eine Transformation von `1 x 8 x 8` zu normalisierten `3 x 64 x 64`-Tensoren.
2. Laden Sie MobileNetV3 Small mit öffentlichen Standardgewichten, falls verfügbar. Verwenden Sie sonst einen dokumentierten Fallback.
3. Frieren Sie alle Feature-Schichten ein und ersetzen Sie die letzte Klassifikationsschicht durch drei Ausgaben.
4. Trainieren Sie nur den Klassifikationskopf für wenige Epochen.
5. Trainieren Sie ein kleines Scratch-CNN auf genau denselben transformierten Bildern und Splits.
6. Vergleichen Sie Validierungs- und Testgenauigkeit, trainierbare Parameter, Median-Inferenzzeit und Größe des `state_dict`.

> **Hinweis:** Zählen Sie nur Parameter mit `requires_grad=True`, wenn Sie den Trainingsaufwand vergleichen.

In [ ]:
transfer_epochs_18 = 1 if FAST_MODE else 3
transfer_batch_size_18 = 32

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integration: MobileNetV3 Small einfrieren und bewerten
#
# Ziel dieser Codezelle:
# Führen Sie ein kleines Transfer-Learning-Projekt für die Klassen 0, 1 und 2 durch.
# 1. Erstellen Sie eine Transformation von 1 x 8 x 8 zu normalisierten 3 x 64 x
# 64-Tensoren. 2. Laden Sie MobileNetV3 Small mit öffentli...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

transfer_epochs_18 = 1 if FAST_MODE else 3
transfer_batch_size_18 = 32

# Die ImageNet-Normalisierung wird auch beim Fallback beibehalten,
# damit beide Architekturläufe dieselben Eingabetensoren erhalten.
transfer_transform_18 = transforms.Compose(
    [
        transforms.Lambda(lambda image: image.repeat(3, 1, 1)),
        transforms.Resize((64, 64), antialias=True),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

transfer_train_dataset_18 = DigitsDataset18(
    TX_train_18,
    Ty_train_18,
    transform=transfer_transform_18,
)
transfer_valid_dataset_18 = DigitsDataset18(
    TX_valid_18,
    Ty_valid_18,
    transform=transfer_transform_18,
)
transfer_test_dataset_18 = DigitsDataset18(
    TX_test_18,
    Ty_test_18,
    transform=transfer_transform_18,
)
transfer_train_loader_18 = DataLoader(
    transfer_train_dataset_18,
    batch_size=transfer_batch_size_18,
    shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
    num_workers=0,
)
transfer_valid_loader_18 = DataLoader(
    transfer_valid_dataset_18,
    batch_size=transfer_batch_size_18,
    shuffle=False,
    num_workers=0,
)
transfer_test_loader_18 = DataLoader(
    transfer_test_dataset_18,
    batch_size=transfer_batch_size_18,
    shuffle=False,
    num_workers=0,
)

requested_weights_18 = None if OFFLINE_MODE else models.MobileNet_V3_Small_Weights.DEFAULT
try:
    transfer_model_18 = models.mobilenet_v3_small(weights=requested_weights_18)
    transfer_initialization_18 = "ImageNet" if requested_weights_18 else "zufällig"
except Exception as download_error:
    print("Vortrainierte Gewichte waren nicht verfügbar. Fallback ohne Download.")
    print("Hinweis:", type(download_error).__name__)
    transfer_model_18 = models.mobilenet_v3_small(weights=None)
    transfer_initialization_18 = "zufällig nach Fallback"

# Feature-Extraktor einfrieren. Der bestehende Klassifikationskopf
# bleibt trainierbar, und seine letzte lineare Schicht wird ersetzt.
for parameter in transfer_model_18.features.parameters():
    parameter.requires_grad = False
input_features_to_last_layer_18 = transfer_model_18.classifier[-1].in_features
transfer_model_18.classifier[-1] = nn.Linear(input_features_to_last_layer_18, 3)
transfer_model_18 = transfer_model_18.to(DEVICE)

transfer_optimizer_18 = torch.optim.Adam(
    [p for p in transfer_model_18.parameters() if p.requires_grad],
    lr=0.001,
)
transfer_criterion_18 = nn.CrossEntropyLoss()

class ScratchCNN18(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 8, 5, stride=2),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, stride=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(16, 3),
        )

    def forward(self, images):
        return self.network(images)

torch.manual_seed(RANDOM_SEED + 2)
scratch_transfer_model_18 = ScratchCNN18().to(DEVICE)
scratch_transfer_optimizer_18 = torch.optim.Adam(
    scratch_transfer_model_18.parameters(),
    lr=0.001,
)

transfer_validation_accuracies_18 = []
scratch_validation_accuracies_18 = []
for epoch in range(transfer_epochs_18):
    train_multiclass_epoch_18(
        transfer_model_18,
        transfer_train_loader_18,
        transfer_criterion_18,
        transfer_optimizer_18,
    )
    _, transfer_valid_accuracy_18, _, _ = evaluate_multiclass_18(
        transfer_model_18,
        transfer_valid_loader_18,
        transfer_criterion_18,
    )
    transfer_validation_accuracies_18.append(transfer_valid_accuracy_18)

    train_multiclass_epoch_18(
        scratch_transfer_model_18,
        transfer_train_loader_18,
        transfer_criterion_18,
        scratch_transfer_optimizer_18,
    )
    _, scratch_valid_accuracy_18, _, _ = evaluate_multiclass_18(
        scratch_transfer_model_18,
        transfer_valid_loader_18,
        transfer_criterion_18,
    )
    scratch_validation_accuracies_18.append(scratch_valid_accuracy_18)

_, transfer_test_accuracy_18, _, _ = evaluate_multiclass_18(
    transfer_model_18,
    transfer_test_loader_18,
    transfer_criterion_18,
)
_, scratch_transfer_test_accuracy_18, _, _ = evaluate_multiclass_18(
    scratch_transfer_model_18,
    transfer_test_loader_18,
    transfer_criterion_18,
)

def trainable_parameter_count_18(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def state_dict_size_bytes_18(model):
    memory_file = io.BytesIO()
    torch.save(model.state_dict(), memory_file)
    return memory_file.getbuffer().nbytes

def median_inference_time_18(model, image_batch, repetitions=3):
    model.eval()
    image_batch = image_batch.to(DEVICE)
    with torch.no_grad():
        _ = model(image_batch)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        durations = []
        for _ in range(repetitions):
            start = time.perf_counter()
            _ = model(image_batch)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            durations.append(time.perf_counter() - start)
    return float(np.median(durations))

timing_images_18, _ = next(iter(transfer_test_loader_18))
transfer_time_18 = median_inference_time_18(transfer_model_18, timing_images_18)
scratch_time_18 = median_inference_time_18(scratch_transfer_model_18, timing_images_18)

transfer_comparison_18 = pd.DataFrame(
    {
        "model": ["MobileNetV3 Small", "Scratch-CNN"],
        "initialization": [transfer_initialization_18, "zufällig"],
        "best_validation_accuracy": [
            max(transfer_validation_accuracies_18),
            max(scratch_validation_accuracies_18),
        ],
        "test_accuracy": [
            transfer_test_accuracy_18,
            scratch_transfer_test_accuracy_18,
        ],
        "trainable_parameters": [
            trainable_parameter_count_18(transfer_model_18),
            trainable_parameter_count_18(scratch_transfer_model_18),
        ],
        "median_inference_seconds": [transfer_time_18, scratch_time_18],
        "state_dict_megabytes": [
            state_dict_size_bytes_18(transfer_model_18) / 1_000_000,
            state_dict_size_bytes_18(scratch_transfer_model_18) / 1_000_000,
        ],
    }
)
print(transfer_comparison_18.round(5).to_string(index=False))

### Reflexion zu Aufgabe 5

Eine eingefrorene Basis verringert die Zahl der zu optimierenden Parameter, nicht aber automatisch Inferenzzeit oder Artefaktgröße. MobileNet ist für mobile Effizienz entwickelt, bleibt aber deutlich größer als ein sehr kleines Scratch-CNN. Vortrainierte ImageNet-Merkmale können bei natürlichen Bildern hilfreich sein; bei stark abweichenden Ziffernbildern ist der Nutzen unsicher. Der Offline-Fallback muss ausdrücklich als nicht vortrainiert dokumentiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.